# Quantifying the Risk–Return Tradeoff in Forecasting

**ArXivist-generated reproduction notebook**  
Paper: https://arxiv.org/abs/2605.09712  
Author: Philippe Goulet Coulombe (Université du Québec à Montréal)  
Generated: 2026-05-12

---

This notebook walks through the key components of the implementation, demonstrates each
risk-adjusted metric with synthetic data, runs a small-scale forecasting evaluation, and
verifies the implementation against the paper's reported results.

**Estimated runtime**: ~5 minutes on CPU (no GPU required for the core metrics).

In [ ]:
# ── Cell 1: Environment Check ──────────────────────────────────────────────
import sys
print(f"Python: {sys.version}")

try:
    import torch
    print(f"PyTorch: {torch.__version__}")
    print(f"CUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
    else:
        print("Running on CPU — NN training will be slower")
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
except ImportError:
    print("PyTorch not installed — metrics and linear models will still work")
    device = None

try:
    import numpy as np
    import pandas as pd
    print(f"NumPy: {np.__version__}")
    print(f"Pandas: {pd.__version__}")
except ImportError as e:
    print(f"[ERROR] Missing dependency: {e}")
    print("Run: pip install -r requirements.txt")

In [ ]:
# ── Cell 2: Installation ───────────────────────────────────────────────────
import subprocess, sys
from pathlib import Path

# Install the package in editable mode
repo_root = Path('.').resolve().parent  # notebooks/ → repo root
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-e', str(repo_root), '--quiet'],
    capture_output=True, text=True
)
if result.returncode == 0:
    print('✓ forecast_risk installed successfully')
else:
    print('[WARNING] Install may have failed:')
    print(result.stderr[-500:])  # Last 500 chars of error

# Add src to path as fallback
src_path = str(repo_root / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)
print(f'Source path: {src_path}')

## Paper Overview

### The Problem

Standard macroeconomic forecast evaluation focuses on **average accuracy** (RMSE, MAE) and
statistical significance (Diebold-Mariano test). This ignores *forecast reliability*: two models
with identical RMSE can have dramatically different patterns of gains and losses over time.

A central bank, finance ministry, or asset manager cannot afford a model that beats the benchmark
by 15% on average but fails catastrophically during the 2008 crisis or 2022 inflation surge.

### The Core Idea

**Treat forecast loss differentials as financial return series** and evaluate them with
risk-adjusted performance measures from finance:

$$r_t = L^B_t - L^M_t$$

where $L^B_t$ is benchmark loss and $L^M_t$ is model loss at time $t$. Positive $r_t$ means
the model beats the benchmark at period $t$.

### Key Metrics

| Metric | Formula | What it captures |
|--------|---------|------------------|
| **Forecast Sharpe** | $\bar{r} / s_r$ | Return per unit of total volatility |
| **Forecast Sortino** | $\bar{r} / s_{\text{down}}$ | Return per unit of *downside* volatility only |
| **Forecast Omega** | $\sum r^+_t / \sum |r^-_t|$ | Total upside vs total downside |
| **Max Drawdown** | $\max_t(M_t - R_t)$ | Worst cumulative underperformance episode |
| **Edge Ratio** | $(\sum e^+/\sum e^-) \times (M-1)$ | Unique information vs forecasting frontier |

### Key Finding

The Survey of Professional Forecasters (SPF) is hard to beat **not because it is the most accurate**,
but because it **rarely fails badly**. Its Sortino and Omega ratios dominate most ML competitors
even when its average RMSE is matched or bettered by machine learning models.

### Implementation Map

```
Paper Section 2.1  → src/forecast_risk/metrics/risk_metrics.py  (returns, Sharpe, Sortino, Omega, MaxDD)
Paper Section 2.4  → src/forecast_risk/metrics/edge_ratio.py    (Edge Ratio)
Paper Section 2.5  → src/forecast_risk/metrics/meta_analysis.py (cross-sectional meta-analysis)
Paper Section 2.3  → src/forecast_risk/utils/dm_test.py         (DM test / Sharpe link)
Paper Section 3    → src/forecast_risk/models/                  (all forecasting models)
Paper Section 3    → src/forecast_risk/evaluation/              (expanding-window engine)
```

## Component 1: Forecast Returns (Section 2.1)

The fundamental building block. Define the **return** at time $t$ as:

$$r_t = L^B_t - L^M_t$$

This transforms forecast evaluation into a financial return series. The average return
$\bar{r}$ corresponds exactly to the standard RMSE/MAE improvement — but now we have the full
distribution $\{r_t\}$ available for risk analysis.

In [ ]:
# ── Cell 5: Forecast Returns Demo ─────────────────────────────────────────
import numpy as np
from forecast_risk.metrics.risk_metrics import compute_returns

np.random.seed(42)
T = 50  # 50 evaluation periods

# Simulate: benchmark is AR(4), model is a ML challenger
losses_benchmark = np.random.exponential(scale=1.5, size=T)  # higher average loss
losses_model     = np.random.exponential(scale=1.0, size=T)  # lower average loss

returns = compute_returns(losses_benchmark, losses_model)

print(f"Return series shape: {returns.shape}")
print(f"Mean return (r_bar): {np.mean(returns):.4f}  [positive = model beats benchmark on average]")
print(f"Periods model wins:  {np.sum(returns > 0)}/{T}")
print(f"Periods model loses: {np.sum(returns < 0)}/{T}")
print(f"\nFirst 10 returns: {returns[:10].round(3)}")

## Component 2: Forecast Sharpe & Sortino Ratios (Section 2.2)

**Forecast Sharpe Ratio** (symmetric risk):
$$\text{Sharpe} = \frac{\bar{r}}{s_r}, \quad s_r^2 = \frac{1}{T-1}\sum_{t=1}^T (r_t - \bar{r})^2$$

**Forecast Sortino Ratio** (downside risk only):
$$\text{Sortino} = \frac{\bar{r}}{s_{\text{down}}}, \quad s_{\text{down}} = \sqrt{\frac{1}{T}\sum_{t=1}^T (\min(r_t, 0))^2}$$

The Sortino ratio is better aligned with forecaster preferences: upside surprises are welcome,
downside failures are costly. The paper's key result is that the **SPF's Sortino ratio
dominates most ML models** even when its Sharpe ratio is comparable.

**Key theoretical link (Section 2.3):**  
When loss differentials are serially uncorrelated:
$$\text{DM} = \sqrt{T} \cdot \text{Sharpe}$$

In [ ]:
# ── Cell 7: Sharpe & Sortino Demo ─────────────────────────────────────────
from forecast_risk.metrics.risk_metrics import ForecastRiskMetrics
from forecast_risk.utils.dm_test import DieboldMarianoTest

rm = ForecastRiskMetrics()

# Paper example: two models with similar RMSE but different risk profiles
np.random.seed(0)
T = 52  # 52 quarters ≈ 13 years

# Model A: moderate gains, high volatility (like a neural network)
r_A = np.concatenate([
    np.random.normal(0.3, 2.0, T//2),   # stable period
    np.random.normal(0.3, 0.5, T//2),   # also moderate
])
# Inject a drawdown episode (simulating financial crisis failure)
r_A[20:25] = -5.0

# Model B: consistent smaller gains, low volatility (like SPF)
r_B = np.random.normal(0.2, 0.3, T)

print(f"{'Metric':<20} {'Model A (NN-like)':>18} {'Model B (SPF-like)':>18}")
print("-" * 58)
print(f"{'Mean Return':<20} {np.mean(r_A):>18.3f} {np.mean(r_B):>18.3f}")
print(f"{'Sharpe':<20} {rm.sharpe(r_A):>18.3f} {rm.sharpe(r_B):>18.3f}")
print(f"{'Sortino':<20} {rm.sortino(r_A):>18.3f} {rm.sortino(r_B):>18.3f}")
print(f"{'Omega':<20} {rm.omega(r_A):>18.3f} {rm.omega(r_B):>18.3f}")
print(f"{'MaxDD':<20} {-rm.max_drawdown(r_A):>18.3f} {-rm.max_drawdown(r_B):>18.3f}")

print("\n→ Model A has similar mean return but is far riskier (paper's central finding!)")
print("→ Model B (SPF-like) dominates on all risk-adjusted metrics despite lower raw return")

# Verify Sharpe ≈ DM/sqrt(T) for i.i.d. returns
dm_B = DieboldMarianoTest(h=1)
losses_b_bench = np.ones(T)
losses_b_model = losses_b_bench - r_B
dm_stat = dm_B.statistic(losses_b_bench, losses_b_model)
sharpe_B = rm.sharpe(r_B)
print(f"\nSharpe/DM link (Model B):")
print(f"  DM / sqrt(T) = {dm_stat / np.sqrt(T):.4f}")
print(f"  Sharpe       = {sharpe_B:.4f}")
print(f"  Ratio        = {(dm_stat / np.sqrt(T)) / sharpe_B:.4f}  (≈ 1.0 for i.i.d. returns)")

## Component 3: Maximum Drawdown (Section 2.2)

Maximum drawdown tracks the worst cumulative underperformance from any historical peak:

$$R_t = \sum_{s=1}^t r_s, \quad M_t = \max_{0 \le u \le t} R_u, \quad \text{MaxDD} = \max_{t} (M_t - R_t)$$

A model may look good on average while experiencing **prolonged episodes of sustained failure** —
the paper highlights the 2021–2022 inflation surge as a prime example where most ML models
failed catastrophically while the SPF remained robust.

In [ ]:
# ── Cell 9: Maximum Drawdown Visualization ─────────────────────────────────
import matplotlib.pyplot as plt

# Use the same r_A and r_B from above
R_A = np.concatenate([[0], np.cumsum(r_A)])
R_B = np.concatenate([[0], np.cumsum(r_B)])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, R, r, label, color in [
    (axes[0], R_A, r_A, 'Model A (NN-like)', 'steelblue'),
    (axes[1], R_B, r_B, 'Model B (SPF-like)', 'goldenrod'),
]:
    M = np.maximum.accumulate(R)
    ax.plot(R, label='Cumulative gain', color=color, lw=2)
    ax.plot(M, label='Running max', color='gray', lw=1, ls='--')
    ax.fill_between(range(len(R)), R, M, alpha=0.2, color='red', label='Drawdown')
    dd = np.max(M - R)
    ax.set_title(f'{label}\nMaxDD = {dd:.2f}')
    ax.set_xlabel('Evaluation period')
    ax.set_ylabel('Cumulative gain $R_t$')
    ax.legend(fontsize=8)
    ax.axhline(0, color='black', lw=0.5)

plt.tight_layout()
plt.suptitle('Maximum Drawdown: Path-Dependent Risk (Paper Section 2.2)', y=1.02, fontsize=11)
plt.savefig('../results/figures/drawdown_demo.png', dpi=120, bbox_inches='tight')
plt.show()
print(f"Model A MaxDD: {rm.max_drawdown(r_A):.3f}  (catastrophic episode at t=20–25)")
print(f"Model B MaxDD: {rm.max_drawdown(r_B):.3f}  (contained downside)")

## Component 4: Edge Ratio (Section 2.4)

The **Edge Ratio** is a novel metric introduced by the paper. It measures whether a model
delivers **unique predictive advantage** relative to the *full forecasting frontier* (the
best-performing alternative at each point in time).

$$L^\star_t = \min_{j \ne M} L_{j,t} \quad (\text{frontier loss at time } t)$$

$$e_{M,t} = L^\star_t - L_{M,t} \quad (\text{positive when M reaches the frontier})$$

$$\text{EdgeRatio}(M) = \frac{\sum_t e^+_{M,t}}{\sum_t e^-_{M,t}} \times (M-1)$$

The $(M-1)$ factor normalizes for pool size. Under the null of equal performance,
$\text{EdgeRatio} \approx 1$. A model that never reaches the frontier gets $\text{EdgeRatio} = 0$.

**Paper finding**: At $h=4$ for GDP, the SPF achieves Edge = 1.10, while at $h=1$ the
Factor-Augmented AR achieves Edge = 0.85 — showing sporadic but unique informational episodes.

In [ ]:
# ── Cell 11: Edge Ratio Demo ───────────────────────────────────────────────
from forecast_risk.metrics.edge_ratio import EdgeRatioCalculator

er = EdgeRatioCalculator()

np.random.seed(42)
M, T = 6, 52  # 6 models, 52 quarters

# Simulate loss matrix [M, T]
losses = np.random.exponential(scale=1.0, size=(M, T))

# Model 0: specialist — usually bad but occasionally excellent
losses[0] = np.random.exponential(1.5, T)  # usually worse
losses[0][10:15] = 0.01                    # brief episodes of uniquely good forecasts

# Model 5: consistent moderate performer (SPF-like)
losses[5] = np.random.exponential(0.9, T)  # consistently lower losses

model_names = ['Specialist (FAAR-like)', 'AR', 'Ridge', 'RF', 'LGB+', 'SPF-like']

print(f"{'Model':<25} {'Mean Loss':>12} {'Edge Ratio':>12} {'Interpretation'}")
print("-" * 75)
edges = er.compute_all(losses)
for i, (name, edge) in enumerate(zip(model_names, edges)):
    mean_loss = np.mean(losses[i])
    interp = "high unique value" if edge > 1.0 else ("moderate" if edge > 0.3 else "rarely frontier")
    print(f"{name:<25} {mean_loss:>12.3f} {edge:>12.3f}   {interp}")

print(f"\nNull expectation under equal performance: EdgeRatio ≈ 1.0")
print(f"Mean across models: {np.mean(edges):.3f}")

## Component 5: Meta-Analysis Metrics (Section 2.5)

For meta-analyses across many targets, horizons, and samples, the same risk-adjusted logic
applies to the **cross-sectional distribution** of percentage improvements:

$$R^M_{v,h,s} = \frac{P^B_{v,h,s} - P^M_{v,h,s}}{P^B_{v,h,s}} \times 100$$

Meta-Sharpe, Meta-Sortino, Meta-Omega, and Meta-Edge ask: *how robustly does this model
outperform across the entire design space?* (Paper Table 1 — GCFK Exercise; Table 2 — M4)

In [ ]:
# ── Cell 13: Meta-Analysis Demo ────────────────────────────────────────────
from forecast_risk.metrics.meta_analysis import MetaAnalysisMetrics, percentage_return

meta = MetaAnalysisMetrics()

np.random.seed(10)
N = 20  # 20 target×horizon×sample combinations

# Simulate RMSE values across design space
perf_dict = {
    'ar4':  np.ones(N),           # benchmark (RMSE ratio = 1)
    'HNN':  np.random.normal(0.88, 0.12, N).clip(0.5, 1.2),  # mostly beats benchmark
    'BART': np.random.normal(0.90, 0.09, N).clip(0.5, 1.2),  # lower variance
    'NN':   np.random.normal(0.87, 0.22, N).clip(0.3, 1.5),  # high variance (sometimes catastrophic)
}

# Compute meta-analysis table
df_meta = meta.full_table(perf_dict, benchmark_key='ar4')

print("Meta-Analysis Table (cross-sectional, like Table 1 in paper):")
print(df_meta.round(3).to_string())
print("\n→ HNN has higher Edge Ratio (decisively best on some targets)")
print("→ BART has lower Vol and higher Sharpe (more consistently good)")
print("→ Paper Table 1: BART Sharpe 0.97 vs HNN 0.85; both Sortino 1.62")

## Component 6: Forecasting Models — AR(4) and Ridge

The paper evaluates 12 models. Here we demonstrate the two linear baselines:

- **AR(4)**: Benchmark — 4 autoregressive lags only
- **Ridge**: High-dimensional linear prediction with L2 regularization, λ cross-validated

In [ ]:
# ── Cell 15: Linear Models Demo ────────────────────────────────────────────
from forecast_risk.models.linear import ARModel, RidgeForecaster

np.random.seed(7)

# Simulate a stationary macro-like series (T=80 quarters, N=30 predictors)
T_total, N = 80, 30
X = np.random.randn(T_total, N)             # predictor panel (standardized)
true_beta = np.random.randn(4) * 0.3        # only AR lags matter
y = X[:, :4] @ true_beta + 0.5 * np.random.randn(T_total)

# Train on first 60, test on last 20
X_train, X_test = X[:60], X[60:]
y_train, y_test = y[:60], y[60:]

# AR(4): uses first 4 columns of X (assumed to be AR lags)
ar = ARModel(lags=4)
ar.fit(X_train, y_train)
ar_preds = ar.predict(X_test)
ar_rmse = np.sqrt(np.mean((y_test - ar_preds)**2))

# Ridge: uses all N=30 predictors with cross-validated lambda
rr = RidgeForecaster(lambda_grid=[0.01, 0.1, 1.0, 10.0], cv_folds=3)
rr.fit(X_train, y_train)
rr_preds = rr.predict(X_test)
rr_rmse = np.sqrt(np.mean((y_test - rr_preds)**2))

print(f"AR(4) test RMSE:  {ar_rmse:.4f}")
print(f"Ridge test RMSE:  {rr_rmse:.4f}")
print(f"RMSE ratio (Ridge/AR): {rr_rmse/ar_rmse:.3f}  (< 1 means Ridge beats AR)")

# Compute risk metrics on out-of-sample losses
ar_losses  = (y_test - ar_preds)**2
rr_losses  = (y_test - rr_preds)**2
returns_rr = compute_returns(ar_losses, rr_losses)

print(f"\nRisk-adjusted metrics for Ridge vs AR(4):")
result = rm.all_metrics(ar_losses, rr_losses, label='Ridge')
for k, v in result.items():
    if isinstance(v, float):
        print(f"  {k:<18}: {v:.4f}")

## Mini-Training: Full Evaluation Pipeline on Synthetic Data

We now run a complete **expanding-window evaluation** on synthetic quarterly data,
mimicking the paper's design:
- Three models: AR(4), Ridge, Random Forest
- 5 evaluation periods (mini-scale; paper uses 50+ quarters)
- Squared error loss
- Risk-adjusted metrics table at the end

In [ ]:
# ── Cell 17: Synthetic Data Setup ─────────────────────────────────────────
np.random.seed(42)

# Simulate 100 quarters of macro-like panel data
T_full = 100
N_series = 20  # smaller panel for speed

# AR(1) target
y_full = np.zeros(T_full)
for t in range(1, T_full):
    y_full[t] = 0.6 * y_full[t-1] + 0.4 * np.random.randn()

# Panel of predictors (lags of y + noise)
X_full = np.zeros((T_full, N_series))
for i in range(min(4, N_series)):      # first 4 cols = lags of y (AR lags)
    X_full[i+1:, i] = y_full[:T_full-i-1]
for i in range(4, N_series):           # remaining = noisy predictors
    X_full[:, i] = 0.2 * y_full + np.random.randn(T_full) * 0.8

print(f"Panel shape:  X={X_full.shape}, y={y_full.shape}")
print(f"Target stats: mean={y_full.mean():.3f}, std={y_full.std():.3f}")
print(f"Simulating {T_full} quarters of quarterly macro data")

In [ ]:
# ── Cell 18: Mini Expanding-Window Evaluation ──────────────────────────────
from forecast_risk.models.linear import ARModel, RidgeForecaster
from forecast_risk.models.tree_models import RandomForestForecaster
from forecast_risk.evaluation.expanding_window import ExpandingWindowEvaluator

# Models to evaluate
models = {
    'ar4':  ARModel(lags=4),
    'ridge': RidgeForecaster(lambda_grid=[0.1, 1.0, 10.0], cv_folds=3),
    'rf':    RandomForestForecaster(n_estimators=50, min_samples_leaf=2),  # smaller for speed
}

evaluator = ExpandingWindowEvaluator(loss_fn_name='squared_error', refit_every=5, verbose=False)

# Evaluation: periods 60–80 (expanding window from t=0)
eval_indices = list(range(60, 80))
print(f"Evaluating on {len(eval_indices)} periods (t=60 to t=79)...")

losses_dict = evaluator.run(
    models=models,
    X=X_full, y=y_full,
    eval_indices=eval_indices,
    horizon=1,
    min_train_size=20,
)

for name, arr in losses_dict.items():
    valid = ~np.isnan(arr)
    print(f"  {name:<10}: mean_loss={np.nanmean(arr):.4f}, n_valid={valid.sum()}")

In [ ]:
# ── Cell 19: Risk-Adjusted Report ─────────────────────────────────────────
from forecast_risk.evaluation.report import RiskAdjustedReport

reporter = RiskAdjustedReport(horizon=1)

try:
    df_report = reporter.generate(losses_dict, benchmark_key='ar4')
    print("Risk-Adjusted Performance Table (vs AR(4) benchmark):")
    print("=" * 60)
    print(df_report.round(3).to_string())
    print("\nColumn guide:")
    print("  Return: mean(r_t) — average % gain over benchmark")
    print("  Sharpe: Return / std(returns)")
    print("  Sortino: Return / downside_std(returns)")
    print("  Omega: sum(r+) / sum(|r-|)")
    print("  MaxDD: worst cumulative loss from peak (negative = bad)")
    print("  Edge: unique advantage vs forecasting frontier")
except Exception as e:
    print(f"[WARNING] Report failed (may be data issue): {e}")
    print("Loss arrays:")
    for k, v in losses_dict.items():
        print(f"  {k}: {v[:5]}")

## Paper Results — Reported Values from the SIR

The following are the key numerical results from the paper that a full reproduction should match.

In [ ]:
# ── Cell 21: Paper Results Reference ──────────────────────────────────────
paper_results = {
    "GDP Growth h=1, Pre-COVID": {
        "SPF Sortino": 1.80,
        "KRR Sortino": 2.85,
        "NN Sortino":  1.09,
        "SPF Edge (h=1)": 0.08,
        "FAAR Edge (h=1)": 0.85,
        "SPF Edge (h=4)": 1.10,
        "note": "SPF dominates on Sortino/Omega despite NN having higher raw return"
    },
    "Unemployment h=1, Pre-COVID": {
        "LGB+ Return": 0.55,
        "LGB+ Sortino": 7.7,
        "TPFN Edge (h=4)": 4.45,
        "note": "LGB+ best risk-adjusted at h=1; TabPFN dominates at h=4"
    },
    "Inflation Pre-2020 h=1": {
        "HNN Sortino": 2.0,
        "SPF Sortino": 1.2,
        "TPFN Edge": 0.86,
        "note": "HNN beats SPF on risk-adjusted metrics; TabPFN has unique frontier wins"
    },
    "Inflation Post-2021 h=1": {
        "HNN Edge": 1.83,
        "HNN is only model with positive return": True,
        "note": "HNN uniquely robust to inflation regime change (key paper result)"
    },
    "Meta-Analysis RMSE (GCFK Table 1)": {
        "HNN Sharpe": 0.85, "BART Sharpe": 0.97,
        "HNN Sortino": 1.62, "BART Sortino": 1.62,
        "HNN Edge": 1.97, "BART Edge": 0.81,
        "note": "BART more consistent; HNN more decisive when it wins"
    },
    "Meta-Analysis Log Score (GCFK Table 1)": {
        "HNN Sharpe": 0.48, "BART Sharpe": -0.41,
        "HNN Edge": 2.59,
        "note": "HNN decisive advantage in density forecasting"
    },
}

print("=" * 65)
print("  Paper Reported Results (arXiv: 2605.09712)")
print("=" * 65)
for setting, metrics in paper_results.items():
    print(f"\n▶ {setting}")
    for k, v in metrics.items():
        print(f"  {k:<40}: {v}")

print("\n" + "=" * 65)
print("To reproduce: run the full pipeline with real FRED-QD + SPF data.")
print("  python run_evaluation.py --config configs/default_config.yaml")
print("  python compute_metrics.py --benchmark ar4 --horizon 1")

## What To Do Next

### Full Reproduction Steps

1. **Download data**:
   ```bash
   python data/download.py --output-dir data/
   ```
   Then manually download SPF medians from the Philadelphia Fed.

2. **Run full evaluation**:
   ```bash
   python run_evaluation.py --config configs/default_config.yaml \
       --targets gdp_growth cpi_inflation unemployment_rate housing_starts \
       --horizons 1 2 4
   ```

3. **Compute metrics** from saved losses:
   ```bash
   python compute_metrics.py --losses-dir results/losses/ --benchmark ar4 --horizon 1
   ```

### Important Implementation Notes (from SIR)

| Assumption | Confidence | Note |
|-----------|-----------|------|
| KRR bandwidth grid: `[0.01, 0.1, 1.0, 10.0, 100.0]` | 55% | Not specified in paper — tune if needed |
| LGB early stopping patience: 50 rounds | 55% | Not specified — adjust via config |
| HNN: **STUB implementation** | — | Requires Goulet Coulombe (2025a, 2026) codebase |
| LGB+/LGBA+: approximated via `linear_tree=True` | — | Requires Goulet Coulombe (2026) codebase |
| HAC NOT applied to Sharpe/Sortino/Omega | 97% | **Deliberate design choice** (paper Sec 2.2) |

### Known Limitations

- **HNN** and **LGB+** are stubs — the actual algorithms require papers not fully published.
- **TabPFN** requires `tabpfn` package with pre-trained weights.
- **SPF** data requires manual download from the Philadelphia Fed.
- Post-COVID results (2021Q1–2024Q2) may differ due to data revisions.

### Resources

- Paper: https://arxiv.org/abs/2605.09712
- FRED-QD: https://research.stlouisfed.org/econ/mccracken/fred-databases/
- SPF: https://www.philadelphiafed.org/surveys-and-data/real-time-data-research/survey-of-professional-forecasters